# 170. SPLADE：学习型稀疏扩展、FLOPS 正则与倒排检索怎样实现？

> **面试问题：SPLADE 与 BM25/稠密检索有什么区别？`log(1+ReLU)`、max pooling、稀疏正则和 impact index 怎样落地？**

## 先给结论

SPLADE 用语言模型词表维度表示 query/document，每个维度仍可进入倒排索引，但非原文词也可获得权重形成语义扩展。常见聚合是对 token 位置的 `log(1+ReLU(logit))` 取 max；训练以排序/蒸馏质量和稀疏正则权衡，线上最终成本由非零维度与 postings 分布决定。

## 推荐回答主线

1. 区分输入 token 与输出词表维度，实现非负饱和变换和 position max pooling。
2. 证明稀疏向量点积可由倒排 postings 累加，解释 learned expansion 如何召回非字面匹配。
3. 实现 ranking/distillation loss 与 FLOPS-style regularizer，并观察稀疏—质量权衡。
4. 覆盖 top-k/阈值、impact 量化、长 posting 热点、ACL、索引版本与 nDCG/延迟评测。

## 教学边界

代码用小型 Embedding+Linear 产生词表 logits，倒排索引是内存字典；不复现预训练 MLM、真实 SPLADE checkpoint、Lucene impact index 或分布式查询执行。

## 一手资料

- [SPLADE](https://arxiv.org/abs/2107.05720)
- [SPLADE v2](https://arxiv.org/abs/2109.10086)
- [Efficient SPLADE](https://arxiv.org/abs/2207.03834)


In [ ]:
import hashlib  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
from collections import defaultdict  # 导入本单元所需的依赖。
from dataclasses import dataclass, asdict  # 导入本单元所需的依赖。

import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。

# 0 是 padding，输出轴是完整词表而非输入序列长度。
torch.manual_seed(170)  # 执行当前语句以推进本节示例。
VOCAB, DIM = 20, 10  # 计算并保存当前步骤的中间状态。
documents = torch.tensor([[1, 2, 3, 0], [4, 5, 0, 0], [1, 6, 7, 0]])  # 计算并保存当前步骤的中间状态。
queries = torch.tensor([[1, 8, 0], [4, 9, 0]])  # 计算并保存当前步骤的中间状态。
d_mask, q_mask = documents.ne(0), queries.ne(0)  # 计算并保存当前步骤的中间状态。

assert documents.max().item() < VOCAB  # 用受控断言验证关键不变量。
assert d_mask.shape == documents.shape  # 用受控断言验证关键不变量。
assert q_mask.sum().item() == 4  # 用受控断言验证关键不变量。


## 1. 手写 SPLADE 表示：非负变换后沿 token 位置取 max

每个输入位置输出 V 维词表 logits，先做 `log1p(ReLU)` 抑制极大权重，再在有效位置取 max。padding 位置要在聚合前归零；空文本应拒绝或返回全零，避免 max 的无定义语义。


In [ ]:
class TinySPLADE(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, vocab, dim):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.embedding = nn.Embedding(vocab, dim, padding_idx=0)  # 计算并保存当前步骤的中间状态。
        self.mlm_head = nn.Linear(dim, vocab)  # 计算并保存当前步骤的中间状态。

    def forward(self, token_ids, mask):  # 定义本节可复用的核心函数。
        logits = self.mlm_head(self.embedding(token_ids))  # 计算并保存当前步骤的中间状态。
        impacts = torch.log1p(torch.relu(logits)) * mask[..., None]  # 计算并保存当前步骤的中间状态。
        return impacts.max(dim=1).values, impacts  # 返回当前分支计算出的结果。

# 输出维度等于词表；权重非负；padding 位置不贡献 impact。
model = TinySPLADE(VOCAB, DIM)  # 计算并保存当前步骤的中间状态。
d_sparse, d_position = model(documents, d_mask)  # 计算并保存当前步骤的中间状态。
q_sparse, q_position = model(queries, q_mask)  # 计算并保存当前步骤的中间状态。
assert d_sparse.shape == (3, VOCAB)  # 用受控断言验证关键不变量。
assert (d_sparse >= 0).all()  # 用受控断言验证关键不变量。
assert d_position[~d_mask].abs().sum().item() == 0.0  # 用受控断言验证关键不变量。


## 2. 学习型扩展：输出词不必在原文出现

MLM head 可以给未出现在文档中的相关词非零权重，这就是 expansion。下面手工 logits 隔离该性质：文档只含 token 1，却在输出词 8 上激活，从而可与只含词 8 的 query 匹配。扩展也会引入漂移和长 postings。


In [ ]:
def splade_pool(logits, mask):  # 定义本节可复用的核心函数。
    transformed = torch.log1p(torch.relu(logits)) * mask[..., None]  # 计算并保存当前步骤的中间状态。
    return transformed.max(1).values  # 返回当前分支计算出的结果。

# 文档输入没有词 8，但输出词表维度 8 可被模型扩展激活。
manual_logits = torch.full((1, 2, VOCAB), -2.0)  # 计算并保存当前步骤的中间状态。
manual_logits[0, 0, 1] = 3.0  # 计算并保存当前步骤的中间状态。
manual_logits[0, 0, 8] = 2.0  # 计算并保存当前步骤的中间状态。
manual_repr = splade_pool(manual_logits, torch.tensor([[1, 0]], dtype=torch.bool))  # 计算并保存当前步骤的中间状态。
assert manual_repr[0, 8] > 0  # 用受控断言验证关键不变量。
assert 8 not in [1]  # 用受控断言验证关键不变量。
assert manual_repr[0, 9] == 0  # 用受控断言验证关键不变量。


## 3. 倒排 index：稀疏点积等于 postings impact 累加

对每个非零词表维度保存 `(doc_id, impact)`，查询只遍历自己的非零维度并累加乘积。数学上等于 dense dot product；工程上成本取决于 query 非零数与每个 term 的 posting 长度。


In [ ]:
def build_impact_index(document_vectors, threshold=0.0):  # 定义本节可复用的核心函数。
    index = defaultdict(list)  # 计算并保存当前步骤的中间状态。
    for doc_id, vector in enumerate(document_vectors):  # 遍历输入元素以累积或检查结果。
        for term in torch.nonzero(vector > threshold, as_tuple=False).flatten().tolist():  # 遍历输入元素以累积或检查结果。
            index[term].append((doc_id, float(vector[term])))  # 执行当前语句以推进本节示例。
    return dict(index)  # 返回当前分支计算出的结果。

def sparse_search(query_vector, index, n_docs, threshold=0.0):  # 定义本节可复用的核心函数。
    scores = np.zeros(n_docs, dtype=float)  # 计算并保存当前步骤的中间状态。
    for term in torch.nonzero(query_vector > threshold, as_tuple=False).flatten().tolist():  # 遍历输入元素以累积或检查结果。
        for doc_id, impact in index.get(term, []):  # 遍历输入元素以累积或检查结果。
            scores[doc_id] += float(query_vector[term]) * impact  # 计算并保存当前步骤的中间状态。
    return scores  # 返回当前分支计算出的结果。

# 倒排累加与完整矩阵乘一致，未知/零 query 返回全零。
impact_index = build_impact_index(d_sparse.detach())  # 计算并保存当前步骤的中间状态。
inverted_scores = sparse_search(q_sparse[0].detach(), impact_index, len(documents))  # 计算并保存当前步骤的中间状态。
dense_scores = (q_sparse[0].detach() @ d_sparse.detach().T).numpy()  # 计算并保存当前步骤的中间状态。
assert np.allclose(inverted_scores, dense_scores)  # 用受控断言验证关键不变量。
assert sparse_search(torch.zeros(VOCAB), impact_index, len(documents)).sum() == 0  # 用受控断言验证关键不变量。
assert all(term < VOCAB for term in impact_index)  # 用受控断言验证关键不变量。


## 4. 排序与蒸馏损失：学习相对相关性，不把 teacher 当真理

可对正负文档做 pairwise softplus，也可拟合 cross-encoder teacher 的软分布。teacher 会携带偏差，训练/验证必须按 query family 切分，并检查 false negative。下面组合 pairwise 与 KL 蒸馏。


In [ ]:
def ranking_distillation_loss(query_vectors, document_vectors, positive_ids, teacher_logits, alpha=0.6):  # 定义本节可复用的核心函数。
    student = query_vectors @ document_vectors.T  # 计算并保存当前步骤的中间状态。
    positive = student[torch.arange(len(query_vectors)), positive_ids]  # 计算并保存当前步骤的中间状态。
    hardest_negative = student.masked_fill(F.one_hot(positive_ids, student.shape[1]).bool(), -torch.inf).max(1).values  # 计算并保存当前步骤的中间状态。
    pairwise = F.softplus(hardest_negative - positive).mean()  # 计算并保存当前步骤的中间状态。
    distill = F.kl_div(F.log_softmax(student, -1), F.softmax(teacher_logits, -1), reduction="batchmean")  # 计算并保存当前步骤的中间状态。
    return alpha * pairwise + (1 - alpha) * distill, student  # 返回当前分支计算出的结果。

# loss 有限并能更新 encoder；正例 id 落在文档范围内。
positive_ids = torch.tensor([0, 1])  # 计算并保存当前步骤的中间状态。
teacher = torch.tensor([[3.0, 0.5, -1.0], [-0.5, 2.5, 0.0]])  # 计算并保存当前步骤的中间状态。
rank_loss, student_scores = ranking_distillation_loss(q_sparse, d_sparse, positive_ids, teacher)  # 计算并保存当前步骤的中间状态。
model.zero_grad(set_to_none=True); rank_loss.backward(retain_graph=True)  # 计算并保存当前步骤的中间状态。
assert torch.isfinite(rank_loss)  # 用受控断言验证关键不变量。
assert positive_ids.max().item() < len(documents)  # 用受控断言验证关键不变量。
assert model.mlm_head.weight.grad is not None  # 用受控断言验证关键不变量。


## 5. FLOPS-style 正则：惩罚 batch 平均激活的平方

常用 surrogate 对每个词表维度的 batch 平均权重平方求和；高频激活维度惩罚更大，间接控制 postings。query/document 可用不同系数。它不是硬 FLOP 计数，真实延迟仍取决于分布和引擎。


In [ ]:
def flops_regularizer(sparse_vectors):  # 定义本节可复用的核心函数。
    mean_activation = sparse_vectors.mean(dim=0)  # 计算并保存当前步骤的中间状态。
    return (mean_activation ** 2).sum()  # 返回当前分支计算出的结果。

# 加倍所有 impact 会让正则放大四倍；全零表示正则为零。
doc_reg = flops_regularizer(d_sparse)  # 计算并保存当前步骤的中间状态。
query_reg = flops_regularizer(q_sparse)  # 计算并保存当前步骤的中间状态。
assert doc_reg >= 0 and query_reg >= 0  # 用受控断言验证关键不变量。
assert torch.allclose(flops_regularizer(2 * d_sparse), 4 * doc_reg)  # 用受控断言验证关键不变量。
assert flops_regularizer(torch.zeros_like(d_sparse)).item() == 0.0  # 用受控断言验证关键不变量。


## 6. 剪枝与 impact 量化：减少 postings，同时监控排序翻转

部署可按 top-k 或阈值剪掉小权重，再把 impact 映射到整数。top-k 给固定每文档上限，阈值更贴合绝对权重但文档间长度不同。量化 scale 必须随索引版本保存。


In [ ]:
def topk_prune(vectors, k):  # 定义本节可复用的核心函数。
    values, ids = torch.topk(vectors, min(k, vectors.shape[1]), dim=1)  # 计算并保存当前步骤的中间状态。
    pruned = torch.zeros_like(vectors)  # 计算并保存当前步骤的中间状态。
    pruned.scatter_(1, ids, values)  # 执行当前语句以推进本节示例。
    return pruned  # 返回当前分支计算出的结果。

def quantize_impacts(vectors, levels=255):  # 定义本节可复用的核心函数。
    scale = vectors.max().clamp_min(1e-8) / levels  # 计算并保存当前步骤的中间状态。
    quantized = torch.round(vectors / scale).clamp(0, levels).to(torch.uint8)  # 计算并保存当前步骤的中间状态。
    return quantized, scale  # 返回当前分支计算出的结果。

# top-k 控制每行非零上限；反量化误差不超过半个 scale（含浮点余量）。
pruned = topk_prune(d_sparse.detach(), k=5)  # 计算并保存当前步骤的中间状态。
impact_q, impact_scale = quantize_impacts(pruned)  # 计算并保存当前步骤的中间状态。
restored = impact_q.float() * impact_scale  # 计算并保存当前步骤的中间状态。
assert (pruned > 0).sum(1).max().item() <= 5  # 用受控断言验证关键不变量。
assert (restored - pruned).abs().max() <= impact_scale / 2 + 1e-6  # 用受控断言验证关键不变量。
assert impact_q.dtype == torch.uint8  # 用受控断言验证关键不变量。


## 7. 效率指标：平均非零数不够，还要看 posting 长尾

少数扩展词若出现在大部分文档，会成为热 posting。报告 document/query 非零数分位、每 term document frequency、每查询访问 postings、压缩字节和 p50/p99；质量则用 Recall/nDCG/MRR 与 lexical/dense baseline 对照。


In [ ]:
def sparsity_report(vectors, threshold=0.0):  # 定义本节可复用的核心函数。
    active = vectors > threshold  # 计算并保存当前步骤的中间状态。
    nnz_per_doc = active.sum(1).cpu().numpy()  # 计算并保存当前步骤的中间状态。
    df = active.sum(0).cpu().numpy()  # 计算并保存当前步骤的中间状态。
    return {  # 返回当前分支计算出的结果。
        "mean_nnz": float(nnz_per_doc.mean()),  # 执行当前语句以推进本节示例。
        "max_nnz": int(nnz_per_doc.max()),  # 执行当前语句以推进本节示例。
        "max_df": int(df.max()),  # 执行当前语句以推进本节示例。
        "total_postings": int(active.sum()),  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。

# 剪枝后总 postings 不增，每词 df 不超过文档数，统计量非负。
before = sparsity_report(d_sparse.detach())  # 计算并保存当前步骤的中间状态。
after = sparsity_report(pruned)  # 计算并保存当前步骤的中间状态。
assert after["total_postings"] <= before["total_postings"]  # 用受控断言验证关键不变量。
assert after["max_df"] <= len(documents)  # 用受控断言验证关键不变量。
assert after["mean_nnz"] >= 0  # 用受控断言验证关键不变量。


## 8. 索引发布：模型与 analyzer/词表/剪枝 recipe 原子绑定

SPLADE 使用固定 tokenizer 词表维度，词表 id 错一位就会查询错误 posting。manifest 还应绑定模型、最大长度、剪枝、impact scale、文档快照和 ACL；迁移期双建索引并做 shadow query。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class SparseIndexManifest:  # 定义承载本节状态与行为的数据结构。
    model_hash: str  # 执行当前语句以推进本节示例。
    tokenizer_hash: str  # 执行当前语句以推进本节示例。
    vocab_size: int  # 执行当前语句以推进本节示例。
    prune_k: int  # 执行当前语句以推进本节示例。
    document_snapshot: str  # 执行当前语句以推进本节示例。
    acl_snapshot: str  # 执行当前语句以推进本节示例。

def manifest_digest(manifest):  # 定义本节可复用的核心函数。
    return hashlib.sha256(json.dumps(asdict(manifest), sort_keys=True).encode()).hexdigest()  # 返回当前分支计算出的结果。

# 在线 query 维度必须匹配 manifest，任何剪枝 recipe 变化都生成新制品。
manifest = SparseIndexManifest("splade-demo-v2", "tok-v5", VOCAB, 5, "docs-42", "acl-9")  # 计算并保存当前步骤的中间状态。
digest = manifest_digest(manifest)  # 计算并保存当前步骤的中间状态。
assert manifest.vocab_size == q_sparse.shape[1]  # 用受控断言验证关键不变量。
assert len(digest) == 64  # 用受控断言验证关键不变量。
assert digest != manifest_digest(SparseIndexManifest("splade-demo-v2", "tok-v5", VOCAB, 8, "docs-42", "acl-9"))  # 用受控断言验证关键不变量。


## 面试收束与生产替换点

完整回答不要停在算法名：先说清业务目标、输入输出与信任边界，再给核心数据结构/公式和可执行 oracle，最后落到离线切片、线上 SLO、成本、安全、版本、灰度与回滚。这里的受控实现用于解释机制和发现反例；真实模型编码器、分布式索引、协议 SDK、安全沙箱、监控与持久层应作为可替换组件，并用同一合同验收。

典型追问包括：数据规模扩大后瓶颈在哪？近似步骤损失了什么？哪个状态必须持久化？超时或部分失败怎样降级？版本错配为何不能静默兼容？离线指标上升是否来自污染、权限泄漏、评测器偏差或重复样本？
